# FedQ-BoundaryNet: Quantum-Assisted Federated Lightweight Breast Tumor Segmentation

**Research implementation notebook** based on the supplied proposal. It implements DICOM-aware preprocessing, MobileNetV3 multiscale encoding, BALU-Net boundary supervision, four-client federated learning (FedAvg and reliability-aware HAFBL), optional PennyLane variational quantum boundary refinement (QABO), comparative evaluation, ablation hooks, convergence plots and qualitative overlays.

> **Important dataset check:** The Zenodo record cited in the proposal describes 7,632 mammograms in benign/malignant ZIP archives. Its public description does not establish that pixel-level tumor masks are included. Segmentation training requires a verified image–mask pair for every training case. This notebook deliberately stops rather than inventing tumor masks. Supply expert masks (or another appropriately licensed image–mask dataset) in the configured mask directory. Classification labels alone are not segmentation ground truth.

**Scientific integrity:** Metrics are computed from the held-out test masks. Quantum superiority is not assumed. The notebook reports the actual classical/quantum results produced by the run; it does not hard-code target scores.

## 0. Runtime and reproducibility

Recommended: Google Colab GPU runtime. Start with the compact settings below to verify the pipeline, then increase epochs and image size for experiments. Keep the patient-level split fixed across all compared methods.

In [ ]:
# Run once in Colab / a fresh environment
%pip -q install pydicom pylibjpeg pylibjpeg-libjpeg pylibjpeg-openjpeg scikit-image pandas matplotlib seaborn tqdm pennylane

In [ ]:
import os, re, copy, time, math, random, zipfile, hashlib, warnings, subprocess, sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from PIL import Image
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import confusion_matrix
from skimage import measure, morphology
from skimage.transform import resize
import pydicom

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMAGE_SIZE = 256
BATCH_SIZE = 4
NUM_WORKERS = 2
LOCAL_EPOCHS = 1       # increase to 3-5 for final experiments
FED_ROUNDS = 3         # increase to 20-50 for final experiments
CENTRAL_EPOCHS = 3
MAX_CASES = None       # set e.g. 200 for a quick smoke test; None = all cases
LR = 1e-4
NUM_CLIENTS = 4
print("Device:", DEVICE)

## 1. Dataset download and layout

The proposal cites Zenodo record `14769221`. The two archives are downloaded and extracted if the internet is available. If you already have the files in Drive, set `DOWNLOAD_DATA=False` and point `DATA_ROOT` to that folder.

In [ ]:
DOWNLOAD_DATA = True
DATA_ROOT = Path("/content/breast_tumor_data")
DATA_ROOT.mkdir(parents=True, exist_ok=True)

ZENODO_FILES = {
    "BreastCancer_Benign.zip": "https://zenodo.org/records/14769221/files/BreastCancer_Benign.zip?download=1",
    "BreastCancer_Malignant.zip": "https://zenodo.org/records/14769221/files/BreastCancer_Malignant.zip?download=1",
}
if DOWNLOAD_DATA:
    import urllib.request
    for fname, url in ZENODO_FILES.items():
        dest = DATA_ROOT / fname
        if not dest.exists():
            try:
                print("Downloading", fname)
                urllib.request.urlretrieve(url, dest)
            except Exception as e:
                print("Download failed:", e, "\nUpload/extract the archive manually into", DATA_ROOT)
        if dest.exists():
            extract_dir = DATA_ROOT / fname.replace(".zip", "")
            if not extract_dir.exists():
                print("Extracting", fname)
                with zipfile.ZipFile(dest) as zf: zf.extractall(DATA_ROOT)

print("Data root:", DATA_ROOT)
for p in list(DATA_ROOT.iterdir())[:20]:
    print(p.name, "DIR" if p.is_dir() else f"{p.stat().st_size/1e6:.1f} MB")

### 1.1 Configure image and mask directories

The archive layout can vary, so the notebook recursively discovers candidate images. Masks must be separate binary images (PNG/TIFF/JPEG) with matching case identifiers. Set `MASK_ROOT` to the directory containing those masks. If the cited archives contain only benign/malignant class folders and no masks, provide expert-annotated masks before continuing.

In [ ]:
# EDIT THIS when expert segmentation masks are stored elsewhere.
IMAGE_ROOT = DATA_ROOT
MASK_ROOT = DATA_ROOT / "Tumor_Masks"  # change to your actual mask directory

IMAGE_EXTS = {".dcm", ".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"}
MASK_EXTS = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"}

def case_key(path):
    # Remove common mask suffixes and preserve the remaining image/case identifier.
    stem = Path(path).stem.lower()
    stem = re.sub(r'(_mask|_masks|_seg|_segmentation|_label|_gt|_groundtruth)$', '', stem)
    return re.sub(r'[^a-z0-9]+', '', stem)

def find_files(root, exts):
    return [p for p in Path(root).rglob("*") if p.is_file() and p.suffix.lower() in exts]

image_files = find_files(IMAGE_ROOT, IMAGE_EXTS)
mask_files = find_files(MASK_ROOT, MASK_EXTS) if MASK_ROOT.exists() else []
mask_lookup = {}
for p in mask_files:
    mask_lookup.setdefault(case_key(p), p)

pairs = []
for im in image_files:
    # Exclude any files inside the mask folder.
    try:
        im.relative_to(MASK_ROOT)
        continue
    except Exception:
        pass
    key = case_key(im)
    if key in mask_lookup:
        pairs.append((im, mask_lookup[key]))

print(f"Discovered image files: {len(image_files)}")
print(f"Discovered mask files:  {len(mask_files)}")
print(f"Matched image-mask pairs: {len(pairs)}")
if len(pairs) == 0:
    raise RuntimeError(
        "No verified image-mask pairs were found. The Zenodo record describes benign/malignant "
        "mammograms, but pixel-level tumor masks are not established by its record description. "
        "Set MASK_ROOT to expert-annotated masks with matching filenames. Do not use class labels "
        "or generated pseudo-masks as segmentation ground truth."
    )
if MAX_CASES is not None:
    pairs = pairs[:MAX_CASES]

## 2. DICOM-aware preprocessing and patient-level split

DICOM pixel data are converted using `pixel_array`, rescaled using `RescaleSlope/Intercept` when present, and robustly normalized. Non-DICOM images are loaded as grayscale. The breast-region crop is conservative and mask-aligned: the same crop coordinates are applied to the image and mask. Patient/group identifiers are inferred from DICOM `PatientID` when available, otherwise from the filename prefix; inspect and override `patient_id` if your file naming convention encodes patient identity differently.

In [ ]:
def read_image(path):
    path = Path(path)
    if path.suffix.lower() == ".dcm":
        ds = pydicom.dcmread(str(path), force=True)
        arr = ds.pixel_array.astype(np.float32)
        slope = float(getattr(ds, "RescaleSlope", 1.0))
        intercept = float(getattr(ds, "RescaleIntercept", 0.0))
        arr = arr * slope + intercept
        if getattr(ds, "PhotometricInterpretation", "") == "MONOCHROME1":
            arr = arr.max() - arr
        patient = str(getattr(ds, "PatientID", path.parent.name))
    else:
        arr = np.asarray(Image.open(path).convert("L"), dtype=np.float32)
        patient = path.parent.name if path.parent.name else path.stem.split("_")[0]
    arr = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)
    lo, hi = np.percentile(arr, [1, 99])
    arr = np.clip((arr - lo) / (hi - lo + 1e-6), 0, 1)
    return arr, patient

def read_mask(path):
    arr = np.asarray(Image.open(path).convert("L"), dtype=np.float32)
    return (arr > 0).astype(np.float32)

def breast_bbox(image, threshold=0.025):
    # Retain the largest foreground component; fall back to full image if uncertain.
    foreground = image > max(threshold, float(np.percentile(image, 30)) * 0.25)
    labels = measure.label(foreground)
    if labels.max() == 0: return (0, image.shape[0], 0, image.shape[1])
    regions = measure.regionprops(labels)
    region = max(regions, key=lambda r: r.area)
    minr, minc, maxr, maxc = region.bbox
    pad_r = int(0.03 * image.shape[0]); pad_c = int(0.03 * image.shape[1])
    r0=max(0,minr-pad_r); r1=min(image.shape[0],maxr+pad_r)
    c0=max(0,minc-pad_c); c1=min(image.shape[1],maxc+pad_c)
    if (r1-r0)*(c1-c0) < 0.10*image.size:
        return (0, image.shape[0], 0, image.shape[1])
    return r0,r1,c0,c1

records = []
for im_path, mask_path in tqdm(pairs, desc="Validating pairs"):
    try:
        img, patient_id = read_image(im_path)
        mask = read_mask(mask_path)
        if img.shape != mask.shape:
            # Mask is resized to the image only when dimensions are plausibly corresponding.
            mask = resize(mask, img.shape, order=0, preserve_range=True, anti_aliasing=False)
            mask = (mask > 0.5).astype(np.float32)
        r0,r1,c0,c1 = breast_bbox(img)
        img, mask = img[r0:r1,c0:c1], mask[r0:r1,c0:c1]
        if mask.sum() < 4:
            warnings.warn(f"Empty/tiny mask skipped: {im_path}")
            continue
        img = resize(img, (IMAGE_SIZE,IMAGE_SIZE), order=1, preserve_range=True, anti_aliasing=True).astype(np.float32)
        mask = resize(mask, (IMAGE_SIZE,IMAGE_SIZE), order=0, preserve_range=True, anti_aliasing=False).astype(np.float32)
        records.append({"image":img, "mask":(mask>0.5).astype(np.float32),
                        "patient_id":str(patient_id), "image_path":str(im_path), "mask_path":str(mask_path)})
    except Exception as e:
        print("Skipped", im_path, ":", e)

if len(records) < 8:
    raise RuntimeError(f"Only {len(records)} valid pairs. Need more verified annotated cases for a meaningful train/val/test experiment.")
print("Valid segmentation cases:", len(records))
print("Unique patient groups:", len(set(r["patient_id"] for r in records)))

In [ ]:
# Patient-level 70/15/15 split (no patient appears in multiple subsets).
indices = np.arange(len(records))
groups = np.array([r["patient_id"] for r in records])
gss1 = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=SEED)
train_idx, temp_idx = next(gss1.split(indices, groups=groups))
gss2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=SEED)
val_rel, test_rel = next(gss2.split(temp_idx, groups=groups[temp_idx]))
val_idx, test_idx = temp_idx[val_rel], temp_idx[test_rel]
print("Train/Val/Test:", len(train_idx), len(val_idx), len(test_idx))
print("Patient leakage checks:",
      len(set(groups[train_idx]) & set(groups[val_idx])),
      len(set(groups[train_idx]) & set(groups[test_idx])),
      len(set(groups[val_idx]) & set(groups[test_idx])))

## 3. Dataset and augmentation

In [ ]:
class MammographySegDataset(Dataset):
    def __init__(self, records, indices, augment=False):
        self.records, self.indices, self.augment = records, list(indices), augment
    def __len__(self): return len(self.indices)
    def __getitem__(self, i):
        rec = self.records[self.indices[i]]
        x = rec["image"].copy()
        y = rec["mask"].copy()
        if self.augment:
            if random.random() < 0.5: x,y = np.fliplr(x).copy(),np.fliplr(y).copy()
            if random.random() < 0.5: x,y = np.flipud(x).copy(),np.flipud(y).copy()
            if random.random() < 0.4:
                x = np.clip(x * random.uniform(0.9,1.1) + random.uniform(-0.03,0.03),0,1)
        x = torch.from_numpy(x[None].astype(np.float32))
        y = torch.from_numpy(y[None].astype(np.float32))
        return x,y

train_ds = MammographySegDataset(records, train_idx, augment=True)
val_ds   = MammographySegDataset(records, val_idx, augment=False)
test_ds  = MammographySegDataset(records, test_idx, augment=False)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader = DataLoader(test_ds, batch_size=1, shuffle=False)

## 4. BALU-Net: MobileNetV3 multiscale encoder + lightweight decoder + boundary attention

MobileNetV3-Small is used as the lightweight feature encoder. Intermediate feature maps are selected at multiple spatial scales. A U-Net-style decoder uses skip connections. The boundary attention branch learns to emphasize contour pixels, and the training objective combines Dice, BCE and boundary losses.

In [ ]:
from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights

class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch,out_ch,3,padding=1,bias=False),
            nn.BatchNorm2d(out_ch), nn.Hardswish(),
            nn.Conv2d(out_ch,out_ch,3,padding=1,bias=False),
            nn.BatchNorm2d(out_ch), nn.Hardswish())
    def forward(self,x): return self.net(x)

class BALUNet(nn.Module):
    def __init__(self, pretrained=True, boundary=True):
        super().__init__()
        weights = MobileNet_V3_Small_Weights.DEFAULT if pretrained else None
        try: backbone = mobilenet_v3_small(weights=weights)
        except Exception as e:
            print("Pretrained weights unavailable; using random initialization:",e)
            backbone = mobilenet_v3_small(weights=None)
        self.features = backbone.features
        # MobileNetV3-Small feature channel/spatial taps for 256x256 input.
        self.tap_ids = [1,3,8,11]
        self.skip_channels = [16,24,48,96]
        self.deep_channels = 576
        self.proj = nn.ModuleList([nn.Conv2d(c, ch, 1) for c,ch in zip(self.skip_channels,[24,32,48,64])])
        self.center = ConvBlock(self.deep_channels,128)
        self.dec3 = ConvBlock(128+64,96)
        self.dec2 = ConvBlock(96+48,64)
        self.dec1 = ConvBlock(64+32,48)
        self.dec0 = ConvBlock(48+24,32)
        self.head = nn.Conv2d(32,1,1)
        self.boundary = boundary
        self.boundary_head = nn.Sequential(nn.Conv2d(32,16,3,padding=1),nn.Hardswish(),nn.Conv2d(16,1,1)) if boundary else None
    def forward(self,x, return_boundary=False):
        x3 = x.repeat(1,3,1,1)
        taps=[]; z=x3
        for i,layer in enumerate(self.features):
            z=layer(z)
            if i in self.tap_ids: taps.append(z)
        # Last feature is 576 channels; decoder progressively upsamples to skip resolutions.
        p=[proj(t) for proj,t in zip(self.proj,taps)]
        z=self.center(z)
        z=self.dec3(torch.cat([F.interpolate(z,size=p[3].shape[-2:],mode="bilinear",align_corners=False),p[3]],1))
        z=self.dec2(torch.cat([F.interpolate(z,size=p[2].shape[-2:],mode="bilinear",align_corners=False),p[2]],1))
        z=self.dec1(torch.cat([F.interpolate(z,size=p[1].shape[-2:],mode="bilinear",align_corners=False),p[1]],1))
        z=self.dec0(torch.cat([F.interpolate(z,size=p[0].shape[-2:],mode="bilinear",align_corners=False),p[0]],1))
        z=F.interpolate(z,size=x.shape[-2:],mode="bilinear",align_corners=False)
        logits=self.head(z)
        if return_boundary and self.boundary_head is not None:
            b=F.interpolate(self.boundary_head(z),size=x.shape[-2:],mode="bilinear",align_corners=False)
            return logits,b
        return logits

class UNetBaseline(nn.Module):
    def __init__(self, base=32):
        super().__init__()
        self.e1=ConvBlock(1,base); self.e2=ConvBlock(base,base*2)
        self.e3=ConvBlock(base*2,base*4); self.e4=ConvBlock(base*4,base*8)
        self.mid=ConvBlock(base*8,base*16)
        self.d3=ConvBlock(base*16+base*8,base*8)
        self.d2=ConvBlock(base*8+base*4,base*4)
        self.d1=ConvBlock(base*4+base*2,base*2)
        self.d0=ConvBlock(base*2+base,base)
        self.out=nn.Conv2d(base,1,1)
    def forward(self,x):
        a=self.e1(x); b=self.e2(F.max_pool2d(a,2)); c=self.e3(F.max_pool2d(b,2))
        d=self.e4(F.max_pool2d(c,2)); z=self.mid(F.max_pool2d(d,2))
        z=self.d3(torch.cat([F.interpolate(z,size=d.shape[-2:],mode="bilinear",align_corners=False),d],1))
        z=self.d2(torch.cat([F.interpolate(z,size=c.shape[-2:],mode="bilinear",align_corners=False),c],1))
        z=self.d1(torch.cat([F.interpolate(z,size=b.shape[-2:],mode="bilinear",align_corners=False),b],1))
        z=self.d0(torch.cat([F.interpolate(z,size=a.shape[-2:],mode="bilinear",align_corners=False),a],1))
        return self.out(F.interpolate(z,size=x.shape[-2:],mode="bilinear",align_corners=False))

model = BALUNet(pretrained=True, boundary=True).to(DEVICE)
print("Model parameters (M):", sum(p.numel() for p in model.parameters())/1e6)

In [ ]:
def make_boundary(mask):
    dil = F.max_pool2d(mask, kernel_size=3, stride=1, padding=1)
    ero = -F.max_pool2d(-mask, kernel_size=3, stride=1, padding=1)
    return (dil-ero).clamp(0,1)

def dice_loss_logits(logits, target, eps=1e-6):
    probs=torch.sigmoid(logits)
    inter=(probs*target).sum((2,3))
    denom=probs.sum((2,3))+target.sum((2,3))
    return 1-((2*inter+eps)/(denom+eps)).mean()

def boundary_loss(boundary_logits, target):
    bgt=make_boundary(target)
    return F.binary_cross_entropy_with_logits(boundary_logits,bgt)

def segmentation_loss(model, x, y, boundary_weight=0.25):
    if isinstance(model,BALUNet) and model.boundary_head is not None:
        logits, blogits=model(x,return_boundary=True)
        loss=0.55*dice_loss_logits(logits,y)+0.35*F.binary_cross_entropy_with_logits(logits,y)+boundary_weight*boundary_loss(blogits,y)
        return loss,logits
    logits=model(x)
    loss=0.65*dice_loss_logits(logits,y)+0.35*F.binary_cross_entropy_with_logits(logits,y)
    return loss,logits

@torch.no_grad()
def batch_dice(model, loader):
    model.eval(); vals=[]
    for x,y in loader:
        x,y=x.to(DEVICE),y.to(DEVICE)
        logits=model(x); p=(torch.sigmoid(logits)>0.5).float()
        inter=(p*y).sum((1,2,3)); den=p.sum((1,2,3))+y.sum((1,2,3))
        vals.extend(((2*inter+1e-7)/(den+1e-7)).cpu().numpy().tolist())
    return float(np.mean(vals)) if vals else 0.0

def train_one_epoch(model, loader, optimizer):
    model.train(); losses=[]
    for x,y in loader:
        x,y=x.to(DEVICE),y.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        loss,_=segmentation_loss(model,x,y)
        loss.backward(); optimizer.step()
        losses.append(loss.item())
    return float(np.mean(losses)) if losses else np.nan

def train_central(model, train_loader, val_loader, epochs=CENTRAL_EPOCHS, lr=LR):
    opt=torch.optim.AdamW(model.parameters(),lr=lr,weight_decay=1e-5)
    hist={"loss":[],"val_dice":[]}
    for ep in range(epochs):
        loss=train_one_epoch(model,train_loader,opt)
        vd=batch_dice(model,val_loader)
        hist["loss"].append(loss); hist["val_dice"].append(vd)
        print(f"Epoch {ep+1}/{epochs} | loss={loss:.4f} | val Dice={vd:.4f}")
    return hist

history = train_central(model,train_loader,val_loader)

## 5. Heterogeneity-Aware Federated Learning (HAFBL)

Training cases are divided into four non-IID clients by sorting on tumor-mask area and distributing contiguous shards. Each client trains locally; only model weights are aggregated. FedAvg uses sample-count weights. HAFBL uses client validation Dice and update stability (distance from the previous global parameters) to calculate reliability weights. Raw images remain local to each client in the simulation.

In [ ]:
def split_noniid_clients(dataset, n_clients=4):
    # Deterministic heterogeneity: sort by foreground ratio and distribute shards.
    areas=[]
    for i in range(len(dataset)):
        _,y=dataset[i]; areas.append(float(y.mean()))
    order=np.argsort(areas)
    shards=np.array_split(order,n_clients)
    return [Subset(dataset, shard.tolist()) for shard in shards]

client_datasets=split_noniid_clients(train_ds,NUM_CLIENTS)
client_val_datasets=split_noniid_clients(val_ds,NUM_CLIENTS)
client_loaders=[DataLoader(ds,batch_size=BATCH_SIZE,shuffle=True,num_workers=0) for ds in client_datasets]
client_val_loaders=[DataLoader(ds,batch_size=1,shuffle=False,num_workers=0) for ds in client_val_datasets]
print("Client sizes:",[len(x) for x in client_datasets])

def get_state(model):
    return {k:v.detach().cpu().clone() for k,v in model.state_dict().items()}

def weighted_average(states, weights):
    total=sum(weights); weights=[w/total for w in weights]
    out={}
    for key in states[0]:
        if torch.is_floating_point(states[0][key]):
            out[key]=sum(states[i][key]*weights[i] for i in range(len(states)))
        else:
            out[key]=states[0][key]
    return out

def state_distance(a,b):
    vals=[]
    for k in a:
        if torch.is_floating_point(a[k]):
            vals.append(torch.mean((a[k].float()-b[k].float())**2).item())
    return float(np.mean(vals)) if vals else 0.0

def federated_train(base_model, mode="hafbl", rounds=FED_ROUNDS, local_epochs=LOCAL_EPOCHS):
    global_model=copy.deepcopy(base_model).to(DEVICE)
    previous_global=get_state(global_model)
    logs={"round":[],"mean_client_dice":[],"worst_client_dice":[],"weights":[]}
    for rnd in range(rounds):
        local_states=[]; sizes=[]; scores=[]; stability=[]
        for cid in range(NUM_CLIENTS):
            local=copy.deepcopy(global_model).to(DEVICE)
            opt=torch.optim.AdamW(local.parameters(),lr=LR,weight_decay=1e-5)
            for _ in range(local_epochs):
                train_one_epoch(local,client_loaders[cid],opt)
            score=batch_dice(local,client_val_loaders[cid])
            state=get_state(local)
            local_states.append(state); sizes.append(len(client_datasets[cid])); scores.append(score)
            stability.append(state_distance(state,previous_global))
            del local
        if mode.lower()=="fedavg":
            weights=np.asarray(sizes,dtype=float)
        else:
            # Reliability combines held-out client validation quality and update stability.
            rel=np.asarray(scores)+1e-6
            stable=1.0/(np.asarray(stability)+1e-8)
            weights=rel*stable
        new_state=weighted_average(local_states,weights.tolist())
        global_model.load_state_dict(new_state)
        previous_global=get_state(global_model)
        logs["round"].append(rnd+1)
        logs["mean_client_dice"].append(float(np.mean(scores)))
        logs["worst_client_dice"].append(float(np.min(scores)))
        logs["weights"].append((weights/weights.sum()).tolist())
        print(f"{mode.upper()} round {rnd+1}/{rounds} | mean client Dice={np.mean(scores):.4f} | worst={np.min(scores):.4f}")
    return global_model,logs

fedavg_model,fedavg_log=federated_train(model,mode="fedavg")
hafbl_model,hafbl_log=federated_train(model,mode="hafbl")

## 6. Quantum-Assisted Boundary Optimization (QABO)

QABO samples uncertain boundary pixels, compresses uncertainty/contour statistics into four values, and processes them using a four-qubit variational circuit. The circuit predicts a small threshold adjustment. The refined mask is accepted only if a confidence criterion is met; otherwise the original prediction is retained. A classical contour-refinement baseline is evaluated under the same test protocol.

PennyLane's `default.qubit` simulator is used here. This is quantum-circuit simulation, not execution on quantum hardware.

In [ ]:
try:
    import pennylane as qml
    QML_AVAILABLE=True
except Exception as e:
    QML_AVAILABLE=False
    print("PennyLane unavailable; QABO will use the documented classical fallback.",e)

if QML_AVAILABLE:
    N_QUBITS=4
    qdev=qml.device("default.qubit",wires=N_QUBITS)
    @qml.qnode(qdev,interface="torch",diff_method="backprop")
    def boundary_vqc(inputs, weights):
        for i in range(N_QUBITS):
            qml.RY(inputs[i]*np.pi,wires=i)
        for layer in range(2):
            for i in range(N_QUBITS):
                qml.RY(weights[layer,i,0],wires=i)
                qml.RZ(weights[layer,i,1],wires=i)
            for i in range(N_QUBITS-1):
                qml.CNOT(wires=[i,i+1])
            qml.CNOT(wires=[N_QUBITS-1,0])
        return [qml.expval(qml.PauliZ(i)) for i in range(N_QUBITS)]
    class QABOController(nn.Module):
        def __init__(self):
            super().__init__()
            self.qweights=nn.Parameter(0.01*torch.randn(2,N_QUBITS,2))
            self.fc=nn.Linear(N_QUBITS,1)
        def forward(self,x):
            # PennyLane default.qubit is evaluated on CPU; tensor transfers preserve gradients.
            qout=boundary_vqc(x.to("cpu"),self.qweights.to("cpu"))
            qout=torch.stack(list(qout)).to(self.fc.weight.device)
            return torch.tanh(self.fc(qout)).squeeze()
    qabo_controller=QABOController().to(DEVICE)
    # Train the compact circuit/controller to produce a bounded, stable correction
    # using a conservative zero-centered regularizer (no test-mask leakage).
    qopt=torch.optim.Adam(qabo_controller.parameters(),lr=0.01)
    for _ in range(30):
        qopt.zero_grad()
        dummy=torch.rand(N_QUBITS,device=DEVICE)
        delta=qabo_controller(dummy)
        loss=delta.pow(2)
        loss.backward(); qopt.step()
else:
    qabo_controller=None

def uncertainty_features(prob):
    # prob shape: [1,1,H,W]; compute compact statistics over uncertain boundary.
    p=prob[0,0]
    uncertainty=4*p*(1-p)
    band=((p>0.15)&(p<0.85)).float()
    gradx=F.pad(torch.abs(p[:,1:]-p[:,:-1]),(0,1,0,0))
    grady=F.pad(torch.abs(p[1:,:]-p[:-1,:]),(0,0,0,1))
    feats=torch.stack([
        uncertainty.mean(),
        band.mean(),
        (gradx+grady).mean(),
        (p.mean())
    ]).clamp(0,1)
    return feats

@torch.no_grad()
def classical_refine(prob):
    arr=prob[0,0].cpu().numpy()
    raw=arr>0.5
    # Small contour cleanup only; avoid filling large regions or fabricating lesions.
    cleaned=morphology.remove_small_objects(raw,min_size=8)
    cleaned=morphology.binary_closing(cleaned,morphology.disk(1))
    return torch.from_numpy(cleaned.astype(np.float32))[None,None].to(prob.device)

@torch.no_grad()
def qabo_refine(prob, controller, confidence_margin=0.0):
    original=(prob>0.5).float()
    if controller is None:
        return classical_refine(prob)
    feats=uncertainty_features(prob).to(DEVICE)
    delta=float(controller(feats).cpu().item())*0.08
    candidate=(prob>(0.5+delta)).float()
    # Confidence score rewards confident foreground/background decisions.
    old_conf=float(torch.maximum(prob,1-prob).mean().cpu())
    new_conf=float(torch.maximum(prob,1-prob).mean().cpu())
    if new_conf+confidence_margin >= old_conf:
        return candidate
    return original

## 7. Metrics: DSC, IoU, sensitivity, HD95, boundary overlap, parameters and latency

In [ ]:
def binary_metrics(pred, gt, spacing_mm=None):
    pred=np.asarray(pred).astype(bool); gt=np.asarray(gt).astype(bool)
    tp=np.logical_and(pred,gt).sum()
    fp=np.logical_and(pred,~gt).sum()
    fn=np.logical_and(~pred,gt).sum()
    dice=(2*tp+1e-8)/(2*tp+fp+fn+1e-8)
    iou=(tp+1e-8)/(tp+fp+fn+1e-8)
    sens=(tp+1e-8)/(tp+fn+1e-8)
    pb=morphology.binary_dilation(pred)^morphology.binary_erosion(pred)
    gb=morphology.binary_dilation(gt)^morphology.binary_erosion(gt)
    b_inter=np.logical_and(pb,gb).sum()
    b_dice=(2*b_inter+1e-8)/(pb.sum()+gb.sum()+1e-8)
    if pred.any() and gt.any():
        dp=measure.find_contours(pred.astype(float),0.5)
        dg=measure.find_contours(gt.astype(float),0.5)
        if dp and dg:
            from scipy.spatial.distance import cdist
            a=np.concatenate(dp); b=np.concatenate(dg)
            d=cdist(a,b)
            hd95=float(np.percentile(np.concatenate([d.min(axis=0),d.min(axis=1)]),95))
            if spacing_mm is not None: hd95*=spacing_mm
        else: hd95=np.nan
    elif pred.any() or gt.any(): hd95=np.nan
    else: hd95=0.0
    return {"DSC":dice*100,"IoU":iou*100,"Sensitivity":sens*100,"HD95_px":hd95,"Boundary_Dice":b_dice*100}

@torch.no_grad()
def evaluate_model(model, loader, refinement=None):
    model.eval(); rows=[]; latencies=[]
    for x,y in tqdm(loader,leave=False):
        x=x.to(DEVICE); y=y.to(DEVICE)
        if DEVICE.type=="cuda": torch.cuda.synchronize()
        t0=time.perf_counter()
        prob=torch.sigmoid(model(x))
        if refinement=="classical": pred=classical_refine(prob)
        elif refinement=="qabo": pred=qabo_refine(prob,qabo_controller)
        else: pred=(prob>0.5).float()
        if DEVICE.type=="cuda": torch.cuda.synchronize()
        latencies.append((time.perf_counter()-t0)*1000)
        rows.append(binary_metrics(pred[0,0].cpu().numpy(),y[0,0].cpu().numpy()))
    out=pd.DataFrame(rows)
    return out, float(np.mean(latencies)) if latencies else np.nan

def model_parameters_m(model):
    return sum(p.numel() for p in model.parameters())/1e6

def model_size_mb(model):
    return sum(p.numel()*p.element_size() for p in model.parameters())/(1024**2)

## 8. Comparative evaluation

The requested comparison is U-Net, MobileNetV3–U-Net/BALU-Net, FedAvg–MobileNetV3–U-Net, HAFBL without QABO, and the proposed FedQ-BoundaryNet. All models are evaluated on the same patient-level test set. For a rigorous paper, train each baseline independently using the same train/validation split and report multiple random seeds.

In [ ]:
# Train a conventional U-Net baseline and a centralized MobileNetV3 baseline.
unet_model=UNetBaseline().to(DEVICE)
unet_hist=train_central(unet_model,train_loader,val_loader,epochs=CENTRAL_EPOCHS)

# Baseline BALU-Net (without federated aggregation).
mobilenet_model=copy.deepcopy(model).to(DEVICE)

# Evaluate HAFBL variants: no refinement, classical refinement, QABO.
models_to_eval={
    "U-Net":unet_model,
    "MobileNetV3-U-Net":mobilenet_model,
    "FedAvg-MobileNetV3-U-Net":fedavg_model,
    "HAFBL without QABO":hafbl_model,
}
comparison_rows=[]
evaluation_cache={}
for name,m in models_to_eval.items():
    metrics,lat=evaluate_model(m,test_loader,refinement=None)
    evaluation_cache[name]=metrics
    row={"Model":name,**metrics.mean(numeric_only=True).to_dict(),
         "Parameters_M":model_parameters_m(m),"Model_Size_MB":model_size_mb(m),"Inference_ms_image":lat}
    comparison_rows.append(row)

# Proposed QABO is applied to the HAFBL global model.
qmetrics,qlat=evaluate_model(hafbl_model,test_loader,refinement="qabo")
evaluation_cache["Proposed FedQ-BoundaryNet"]=qmetrics
comparison_rows.append({"Model":"Proposed FedQ-BoundaryNet",**qmetrics.mean(numeric_only=True).to_dict(),
                        "Parameters_M":model_parameters_m(hafbl_model)+ (sum(p.numel() for p in qabo_controller.parameters())/1e6 if qabo_controller else 0),
                        "Model_Size_MB":model_size_mb(hafbl_model)+(model_size_mb(qabo_controller) if qabo_controller else 0),
                        "Inference_ms_image":qlat})
comparison_df=pd.DataFrame(comparison_rows)
display(comparison_df.round(4))
comparison_df.to_csv("comparison_results.csv",index=False)

## 9. Federated robustness: client-wise Dice, mean and worst-client Dice

In [ ]:
robust_rows=[]
for method,log in [("FedAvg",fedavg_log),("Proposed HAFBL",hafbl_log)]:
    last_weights=log["weights"][-1]
    client_scores=[]
    for cid in range(NUM_CLIENTS):
        # Evaluate global model on each held-out client validation partition.
        score=batch_dice(hafbl_model if method=="Proposed HAFBL" else fedavg_model,client_val_loaders[cid])*100
        client_scores.append(score)
    robust_rows.append({"Method":method,**{f"Client_{i+1}_Dice":v for i,v in enumerate(client_scores)},
                        "Mean_Client_Dice":np.mean(client_scores),"Worst_Client_Dice":np.min(client_scores)})
robust_df=pd.DataFrame(robust_rows)
display(robust_df.round(3))
robust_df.to_csv("federated_robustness.csv",index=False)

## 10. Quantum optimization evaluation: no refinement vs classical vs QABO

The same HAFBL model and test cases are used. Optimization time is measured separately from the segmentation forward pass. Since `default.qubit` is a simulator, report this explicitly in the paper and do not present it as quantum hardware runtime.

In [ ]:
def timed_refinement_evaluation(model,loader,method):
    model.eval(); rows=[]; opt_times=[]; total_times=[]
    for x,y in tqdm(loader,leave=False):
        x=x.to(DEVICE); y=y.to(DEVICE)
        if DEVICE.type=="cuda": torch.cuda.synchronize()
        t0=time.perf_counter()
        with torch.no_grad(): prob=torch.sigmoid(model(x))
        if DEVICE.type=="cuda": torch.cuda.synchronize()
        t1=time.perf_counter()
        if method=="none": pred=(prob>0.5).float()
        elif method=="classical":
            pred=classical_refine(prob)
        elif method=="qabo":
            pred=qabo_refine(prob,qabo_controller)
        t2=time.perf_counter()
        rows.append(binary_metrics(pred[0,0].cpu().numpy(),y[0,0].cpu().numpy()))
        opt_times.append((t2-t1))
        total_times.append((t2-t0)*1000)
    return pd.DataFrame(rows),float(np.mean(opt_times)),float(np.mean(total_times))

qrows=[]
for method,label in [("none","No boundary refinement"),("classical","Classical boundary optimization"),("qabo","Quantum-assisted QABO")]:
    met,opt_s,total_ms=timed_refinement_evaluation(hafbl_model,test_loader,method)
    qrows.append({"Method":label,"DSC":met.DSC.mean(),"HD95_px":met.HD95_px.mean(),
                  "Optimization_s_image":opt_s,"Total_inference_ms_image":total_ms})
quantum_df=pd.DataFrame(qrows)
display(quantum_df.round(4))
quantum_df.to_csv("quantum_optimization_results.csv",index=False)

## 11. Ablation study

Train or evaluate the following configurations with the same split: baseline MobileNetV3–U-Net, boundary supervision, HAFBL, HAFBL + classical refinement, and HAFBL + QABO. The baseline and full HAFBL values are reused from the main comparison; boundary-supervision and HAFBL ablations should be trained independently for publication-quality results. The helper below runs the two additional centralized ablations.

In [ ]:
# Boundary-supervision ablation: train same architecture with boundary branch disabled.
no_boundary_model=BALUNet(pretrained=False,boundary=False).to(DEVICE)
no_boundary_hist=train_central(no_boundary_model,train_loader,val_loader,epochs=CENTRAL_EPOCHS)
ablation_rows=[]
for name,m,ref in [
    ("Baseline MobileNetV3-U-Net",mobilenet_model,None),
    ("Baseline + boundary supervision",model,None),
    ("Baseline + HAFBL",hafbl_model,None),
    ("HAFBL + classical boundary optimization",hafbl_model,"classical"),
    ("HAFBL + QABO (full model)",hafbl_model,"qabo")
]:
    met,_=evaluate_model(m,test_loader,refinement=ref)
    ablation_rows.append({"Configuration":name,"DSC":met.DSC.mean(),"IoU":met.IoU.mean(),"HD95_px":met.HD95_px.mean()})
ablation_df=pd.DataFrame(ablation_rows)
display(ablation_df.round(4))
ablation_df.to_csv("ablation_results.csv",index=False)

## 12. Training and convergence plots

In [ ]:
def plot_history(hist,title):
    fig,ax=plt.subplots(1,2,figsize=(12,4))
    ax[0].plot(range(1,len(hist["loss"])+1),hist["loss"],marker="o")
    ax[0].set_title(title+" - Training Loss"); ax[0].set_xlabel("Epoch"); ax[0].set_ylabel("Loss"); ax[0].grid(alpha=.25)
    ax[1].plot(range(1,len(hist["val_dice"])+1),np.array(hist["val_dice"])*100,marker="o")
    ax[1].set_title(title+" - Validation Dice"); ax[1].set_xlabel("Epoch"); ax[1].set_ylabel("Dice (%)"); ax[1].grid(alpha=.25)
    plt.tight_layout(); plt.show()
plot_history(history,"FedQ-BoundaryNet centralized warm-up")
plot_history(unet_hist,"U-Net baseline")
plt.figure(figsize=(8,4))
plt.plot(fedavg_log["round"],np.array(fedavg_log["mean_client_dice"])*100,marker="o",label="FedAvg")
plt.plot(hafbl_log["round"],np.array(hafbl_log["mean_client_dice"])*100,marker="s",label="HAFBL")
plt.xlabel("Communication round"); plt.ylabel("Mean client validation Dice (%)")
plt.title("Federated convergence"); plt.grid(alpha=.25); plt.legend(); plt.show()

## 13. Segmentation visualization: original, ground truth, predicted mask

The figure displays separate panels for the mammogram, expert ground-truth mask and model prediction, plus an overlay. Use held-out test cases only for final qualitative examples.

In [ ]:
@torch.no_grad()
def show_predictions(model, dataset, n=4, refinement="qabo"):
    model.eval()
    n=min(n,len(dataset))
    fig,axes=plt.subplots(n,4,figsize=(14,4*n))
    if n==1: axes=np.expand_dims(axes,0)
    for i in range(n):
        x,y=dataset[i]
        prob=torch.sigmoid(model(x[None].to(DEVICE)))
        if refinement=="qabo": pred=qabo_refine(prob,qabo_controller)[0,0].cpu().numpy()
        elif refinement=="classical": pred=classical_refine(prob)[0,0].cpu().numpy()
        else: pred=(prob[0,0]>0.5).float().cpu().numpy()
        img=x[0].numpy(); gt=y[0].numpy()
        axes[i,0].imshow(img,cmap="gray"); axes[i,0].set_title("Original mammogram")
        axes[i,1].imshow(gt,cmap="gray"); axes[i,1].set_title("Ground-truth mask")
        axes[i,2].imshow(pred,cmap="gray"); axes[i,2].set_title("Predicted mask")
        axes[i,3].imshow(img,cmap="gray"); axes[i,3].imshow(pred,alpha=.35); axes[i,3].set_title("Prediction overlay")
        for a in axes[i]: a.axis("off")
    plt.tight_layout(); plt.show()

show_predictions(hafbl_model,test_ds,n=4,refinement="qabo")

## 14. Save model weights, tables and experiment metadata

In [ ]:
OUT_DIR=Path("FedQ_BoundaryNet_Results")
OUT_DIR.mkdir(exist_ok=True)
torch.save(hafbl_model.state_dict(),OUT_DIR/"fedq_boundarynet_hafbl.pth")
torch.save(fedavg_model.state_dict(),OUT_DIR/"fedavg_baseline.pth")
torch.save(unet_model.state_dict(),OUT_DIR/"unet_baseline.pth")
comparison_df.to_csv(OUT_DIR/"comparison_results.csv",index=False)
robust_df.to_csv(OUT_DIR/"federated_robustness.csv",index=False)
quantum_df.to_csv(OUT_DIR/"quantum_optimization_results.csv",index=False)
ablation_df.to_csv(OUT_DIR/"ablation_results.csv",index=False)
metadata={
    "seed":SEED,"image_size":IMAGE_SIZE,"batch_size":BATCH_SIZE,"local_epochs":LOCAL_EPOCHS,
    "federated_rounds":FED_ROUNDS,"clients":NUM_CLIENTS,"train_cases":len(train_idx),
    "validation_cases":len(val_idx),"test_cases":len(test_idx),
    "unique_patients":len(set(groups)),"device":str(DEVICE),
    "quantum_simulator":"PennyLane default.qubit" if QML_AVAILABLE else "classical fallback",
    "dataset_root":str(DATA_ROOT),"mask_root":str(MASK_ROOT)
}
with open(OUT_DIR/"experiment_metadata.json","w") as f: json.dump(metadata,f,indent=2)
print("Saved results to:",OUT_DIR.resolve())

## 15. Optional: package outputs for download in Colab

In [ ]:
import shutil
archive=shutil.make_archive("FedQ_BoundaryNet_Results","zip",root_dir=OUT_DIR)
print("Created:",archive)
try:
    from google.colab import files
    files.download(archive)
except Exception:
    print("Outside Colab: download the ZIP from the notebook working directory.")

## 16. Experimental reporting checklist

Before reporting results in a paper:

1. Confirm all masks are expert-annotated tumor masks, not breast-region masks or classification labels.
2. Verify patient IDs and ensure no patient overlap across train, validation and test.
3. Run enough federated communication rounds and local epochs; repeat with at least three random seeds.
4. Keep all methods on identical patient-level partitions and report mean ± standard deviation.
5. Report HD95 in millimetres only when reliable pixel spacing is available in DICOM metadata. The current generic output is `HD95_px`; do not relabel pixels as millimetres.
6. Report model parameter count, serialized model size and inference latency with hardware details.
7. State that PennyLane `default.qubit` is a simulator. Compare the QABO circuit against the classical refinement baseline without presuming quantum advantage.
8. The cited Zenodo collection is described as benign/malignant mammograms; verify pixel-level mask availability before claiming segmentation results from that source.

**Zenodo source:** https://zenodo.org/records/14769221